In [14]:
!pip install duckdb

zsh:1: /Users/keerthana/InfraVis/ShipAhoy_KJ/shipahoy/bin/pip: bad interpreter: /Users/keerthana/InfraVis/ShipAhoy/shipahoy/bin/python: no such file or directory
/opt/homebrew/Cellar/pyenv/2.6.32/pyenv.d/exec/pip-rehash/pip: /Users/keerthana/InfraVis/ShipAhoy_KJ/shipahoy/bin/pip: /Users/keerthana/InfraVis/ShipAhoy/shipahoy/bin/python: bad interpreter: No such file or directory


In [15]:
import duckdb

con = duckdb.connect()

In [16]:
# Scrapping the Python function for h3 computations in this version
# Install and load the native DuckDB extension for h3

con.execute("INSTALL h3 FROM community;")
con.execute("LOAD h3;")

In [17]:
query = """
COPY (
WITH ordered AS (
    SELECT
        mmsi,
        date_time_utc,
        lat,
        lon,
        ais_ship_type,
        LEAD(date_time_utc) OVER (PARTITION BY mmsi ORDER BY date_time_utc) - date_time_utc AS dt -- Calculate consecutive time differences within a given MMSI 
    FROM read_parquet('/Users/keerthana/InfraVis/ShipAhoy_KJ/data/*.parquet')
),

cleaned AS (
    SELECT *,
        EXTRACT(EPOCH FROM dt)/60.0 AS time_diff_min -- Calculate the minutes from timestamp format
    FROM ordered
    WHERE dt IS NOT NULL
),

capped AS (
    SELECT *,
        CASE
            WHEN time_diff_min > 15 THEN 15 -- Capping values over 15 minutes as normalization, prevents docking time from overcrowding the density
            ELSE time_diff_min
        END AS tmin
    FROM cleaned
),

ship_grouped AS ( -- Classify ship types
    SELECT *,
        CASE
            WHEN ais_ship_type BETWEEN 60 AND 69 THEN 'Passenger'
            WHEN ais_ship_type BETWEEN 70 AND 79 THEN 'Cargo'
            WHEN ais_ship_type BETWEEN 80 AND 89 THEN 'Tanker'
            ELSE NULL
        END AS ship_type
    FROM capped
    WHERE ship_type IS NOT NULL
),

with_h3 AS ( 
    SELECT *,
        h3_latlng_to_cell_string(lat, lon, 4)::VARCHAR AS h3_cell -- DuckDB's native extension to convert lat/lon to h3 cells in string format
    FROM ship_grouped
)

SELECT
    h3_cell,
    SUM(CASE WHEN ship_type='Passenger' THEN ROUND(tmin)::INTEGER ELSE 0 END) AS countPassenger,  
    SUM(CASE WHEN ship_type='Cargo' THEN ROUND(tmin)::INTEGER ELSE 0 END) AS countCargo,
    SUM(CASE WHEN ship_type='Tanker' THEN ROUND(tmin)::INTEGER ELSE 0 END) AS countTanker
FROM with_h3
GROUP BY h3_cell
ORDER BY h3_cell -- Add up all the time differences for each ship type in each h3 cell 
) TO '/Users/keerthana/InfraVis/ShipAhoy_KJ/results/shipDensityCounts.csv' (HEADER, DELIMITER ',');
"""

In [18]:
con.execute(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))